In [3]:
import csv
import hashlib
import numpy as np
from gostcrypto import gosthash
from collect import collect
from nist_sts.run_tests import *

# Extract

In [4]:
collect(5000000, 'data.csv')

Найден порт: COM5 - Устройство с последовательным интерфейсом USB (COM5)
Сбор 5,000,000 отсчетов... (Ctrl+C для остановки)
  Прогресс: 5,000,192/5,000,000 (100%)

Собрано: 5,000,000 отсчетов
Время: 24.0 сек
Скорость: 208,749 отсчетов/сек
Файл сохранён: data.csv


In [5]:
def load_data(filename):
    data = []
    with open(filename, 'r') as f:
        reader = csv.reader(f)
        next(reader, None)
        for row in reader:
            if row and row[1].isdigit():
                data.append(int(row[1]))

    print(f"Загружено {len(data)} отсчетов.")
    return data

In [ ]:
def extract_compressed(raw_data, target_bits, hash_type="sha3_256", compression_ratio=2):
    hash256 = ['sha256', 'sha3_256', 'blake2s', 'streebog256']
    hash512 = ['sha512', 'sha3_512', 'blake2b', 'streebog512']

    if hash_type in hash256:
        input_block_size = 256
        hash_format = '0256b'
    elif hash_type in hash512:
        input_block_size = 512 
        hash_format = '0512b'
    else:
         raise ValueError()
    
    diffs = np.diff(raw_data)
    raw_bits = (diffs & 1).astype(int)
    bits_list = raw_bits.tolist()

    bit_string = "".join(map(str, bits_list))
    
    output_bits_per_hash = int(input_block_size // compression_ratio)
    
    final_bits = []

    for i in range(0, len(bit_string) - input_block_size, input_block_size):
        if len(final_bits) >= target_bits:
            break
            
        block_str = bit_string[i : i + input_block_size]
        if not all(c in '01' for c in block_str):
            continue
            
        block_int = int(block_str, 2)
        block_bytes = block_int.to_bytes(input_block_size // 8, byteorder='big')

        if hash_type == "sha3_512":
            hash_obj = hashlib.sha3_512(block_bytes)
        elif hash_type == "sha3_256":
            hash_obj = hashlib.sha3_256(block_bytes)
        elif hash_type == "sha512":
            hash_obj = hashlib.sha512(block_bytes)
        elif hash_type == "sha256":
            hash_obj = hashlib.sha256(block_bytes)

        elif hash_type == "blake2b":
            hash_obj = hashlib.blake2b(block_bytes, digest_size=64)
        elif hash_type == "blake2s":
            hash_obj = hashlib.blake2s(block_bytes, digest_size=32)

        elif hash_type == "streebog512":
            hash_obj = gosthash.new('streebog512', data=block_bytes)
        elif hash_type == "streebog256":
            hash_obj = gosthash.new('streebog256', data=block_bytes)

        hash_hex = hash_obj.hexdigest()
        hash_int = int(hash_hex, 16)
        hash_bin = format(hash_int, hash_format) 
        
        for b in hash_bin[:output_bits_per_hash]:
            if len(final_bits) >= target_bits:
                break
            final_bits.append(int(b))
            
    return np.array(final_bits, dtype=int)

In [27]:
data = load_data('data.csv')
extracted_bits = extract_compressed(data, 1000000, 'shake_128', 3)
run_tests(extracted_bits)

Загружено 5000000 отсчетов.


TypeError: hexdigest() missing required argument 'length' (pos 1)

In [16]:
data = load_data('data.csv')
extracted_bits = extract_compressed(data, 1000000, 'blake2s', 3)
run_tests(extracted_bits)

Загружено 5000000 отсчетов.
Длина последовательности: 1000000 бит
Распределение: 0.5007 (1) / 0.4993 (0)
----------------------------------------------------------------------
Название теста                                | P-value    | Статус
----------------------------------------------------------------------
1. Frequency (Monobit) Test                   | 0.140482   | Пройден
2. Block Frequency Test                       | 0.116275   | Пройден
3. Runs Test                                  | 0.286501   | Пройден
4. Longest Run of Ones in a Block             | 0.364402   | Пройден
5. Binary Matrix Rank Test                    | 0.813178   | Пройден
6. Spectral (DFT) Test                        | 0.406031   | Пройден
7. Non-overlapping Template (mean P)          | 0.724094   | Пройден
8. Overlapping Template Matching (m=9)        | 0.343144   | Пройден
9. Maurer's Universal Test (L=6)              | 0.982209   | Пройден
10. Linear Complexity Test                    | 0.081026   | Про

In [ ]:
data = load_data('data.csv')
extracted_bits = extract_compressed(data, 1000000, 'streebog256', 3)
run_tests(extracted_bits)

Загружено 5000000 отсчетов.
Длина последовательности: 996081 бит
Распределение: 0.4999 (1) / 0.5001 (0)
----------------------------------------------------------------------
Название теста                                | P-value    | Статус
----------------------------------------------------------------------
1. Frequency (Monobit) Test                   | 0.778287   | Пройден
2. Block Frequency Test                       | 0.853248   | Пройден
3. Runs Test                                  | 0.103467   | Пройден
4. Longest Run of Ones in a Block             | 0.594970   | Пройден
5. Binary Matrix Rank Test                    | 0.890569   | Пройден
6. Spectral (DFT) Test                        | 0.545195   | Пройден
7. Non-overlapping Template (mean P)          | 0.544075   | Пройден
Предупреждение: Overlapping Template Test требует минимум 10^6 бит для надежности. Получено 996081. Результаты могут быть неточными
8. Overlapping Template Matching (m=9)        | 0.349588   | Пройден
9.